# MORL-PCT Pareto-front evaluation — Kaggle

Sweeps a preference grid across the 5-dimensional simplex, runs the trained MORL
policy under each preference on a fixed voyage set, and visualises the resulting
Pareto frontiers in 2D pairs and a 3D triple. This is the **headline figure for
the paper** — heuristics give one operating point each; the MORL policy traces a
surface.

Produces:
- `pareto_grid.csv` — every (preference, voyage, algorithm) result
- `pareto_2d_*.png` — 2D Pareto fronts for each pair of objectives
- `pareto_3d.png` — 3D scatter for (util, access, stability)
- `multi_metric_summary.csv` — table for the paper
- `eval_morl_results.zip`

## 1. Setup

In [ ]:
import os, sys, subprocess, glob, shutil
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
REPO_URL = 'https://github.com/Seif-Sameh/loading-service-2.git'
BRANCH = 'main'
if os.path.isdir('loading-service-2'):
    subprocess.run(['rm', '-rf', 'loading-service-2'], check=True)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, 'loading-service-2'], check=True)
os.chdir('loading-service-2')
if os.getcwd() not in sys.path: sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--upgrade-strategy', 'only-if-needed',
                '-r', 'requirements.txt', '-r', 'requirements-rl.txt'], check=True)
if not os.path.isdir('/tmp/wadaboa-bpp'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Wadaboa/3d-bpp.git', '/tmp/wadaboa-bpp'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_datasets',
                '--wadaboa-pkl', '/tmp/wadaboa-bpp/data/products.pkl'], check=True)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
# Locate the trained MORL checkpoint
cands = sorted(glob.glob('/kaggle/input/**/morl_pct_latest.pt', recursive=True))
if not cands:
    cands = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
if not cands:
    raise FileNotFoundError('Upload morl_pct_latest.pt as a Kaggle dataset first.')
MODEL_PATH = cands[0]
print('using model:', MODEL_PATH, f'({os.path.getsize(MODEL_PATH)/1024:.1f} KB)')

In [ ]:
import time, random, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from app.catalog.loader import get_container
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig
from app.algorithms import get_algorithm
from app.algorithms.base import solve
from app.algorithms.pct.morl_agent import MORLPCTAgent

OUT = Path('eval_morl_out'); OUT.mkdir(parents=True, exist_ok=True)
HEURISTICS = ['bl', 'extreme_points', 'baf', 'bssf', 'blsf']

## 2. Build the preference grid

In [ ]:
OBJ_LABELS = ['util', 'access', 'stab', 'cog', 'lifo']

def build_preference_grid() -> list[list[float]]:
    """30 representative preference vectors covering the 5-simplex:
       - 5 single-objective corners
       - 10 pairwise (50/50) edges
       - 10 triple (33/33/33) faces
       - 1 uniform centre
       - 4 utility-leaning blends (most operationally common)
    """
    grid = []
    n = len(OBJ_LABELS)
    # 5 corners
    for i in range(n):
        v = [0.0]*n; v[i] = 1.0; grid.append(v)
    # 10 pairs
    for i, j in itertools.combinations(range(n), 2):
        v = [0.0]*n; v[i] = v[j] = 0.5; grid.append(v)
    # 10 triples
    for i, j, k in itertools.combinations(range(n), 3):
        v = [0.0]*n; v[i] = v[j] = v[k] = 1/3; grid.append(v)
    # uniform
    grid.append([1.0/n]*n)
    # utility-leaning operational blends
    grid += [
        [0.7, 0.1, 0.1, 0.05, 0.05],   # util-heavy
        [0.4, 0.4, 0.1, 0.05, 0.05],   # bulk-shipping (util + access)
        [0.3, 0.1, 0.3, 0.3, 0.0],     # marine-shipping (util + stab + CoG)
        [0.2, 0.5, 0.0, 0.0, 0.3],     # last-mile delivery (access + LIFO)
    ]
    return grid

PREF_GRID = build_preference_grid()
print(f'preference grid size: {len(PREF_GRID)}')
for i, p in enumerate(PREF_GRID[:8]):
    print(f'  {i:>2}: {[round(x,2) for x in p]}')

## 3. Voyage set + per-step metric helpers

In [ ]:
N_VOYAGES = 15        # bump to 30+ for thesis-grade variance
ITEMS = 60
alex = AlexandriaSampler(SamplerConfig(n_items=ITEMS, strategy='mixed', seed=None))
VOYAGES = [(get_container('40HC'), alex.sample()) for _ in range(N_VOYAGES)]
print(f'evaluation voyage set: {N_VOYAGES} voyages x {ITEMS} items on 40HC')

def metrics(res, container, items_by_id):
    placed = res.placements
    util = res.kpis.utilization
    placed_pct = len(placed) / max(len(items_by_id), 1)
    L = container.internal.length_mm
    door = 0.20 * L
    if placed:
        access = sum(
            1 for p in placed
            if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
        ) / len(placed)
        stab = max(0.0, (len(placed) - res.kpis.unstable_count) / len(placed))
    else:
        access, stab = 0.0, 0.0
    cog_balance = 1.0 - (abs(res.kpis.cog_long_dev) + abs(res.kpis.cog_lat_dev))
    lifo = (len(placed) - res.kpis.lifo_violation_count) / max(len(placed), 1)
    return {
        'util': util, 'placed_pct': placed_pct,
        'access': access, 'stab': stab,
        'cog_balance': cog_balance, 'lifo': lifo,
        'elapsed_ms': res.elapsed_ms,
    }

## 4. Run the preference sweep

In [ ]:
rows = []

# Baselines: each heuristic is ONE preference-agnostic point
for hcode in HEURISTICS:
    h = get_algorithm(hcode)
    for vi, (c, its) in enumerate(VOYAGES):
        items_by_id = {it.id: it for it in its}
        res, _ = solve(algorithm=h, container=c, items=its)
        m = metrics(res, c, items_by_id)
        rows.append({'algorithm': hcode, 'pref_idx': -1, 'preference': None, 'voyage': vi, **m})
    print(f'heuristic {hcode}: done')

# MORL: one agent per preference
for pi, pref in enumerate(PREF_GRID):
    agent = MORLPCTAgent(weights_path=MODEL_PATH, preference=pref, device=device)
    for vi, (c, its) in enumerate(VOYAGES):
        items_by_id = {it.id: it for it in its}
        res, _ = solve(algorithm=agent, container=c, items=its)
        m = metrics(res, c, items_by_id)
        rows.append({'algorithm': 'morl_pct', 'pref_idx': pi, 'preference': pref, 'voyage': vi, **m})
    if pi % 5 == 0:
        print(f'  pref {pi}/{len(PREF_GRID)} done')

df = pd.DataFrame(rows)
df.to_csv(OUT/'pareto_grid.csv', index=False)
print(f'\nwrote {OUT/"pareto_grid.csv"}: {len(df)} rows')

## 5. Pareto frontier — 2D scatter for each pair of objectives

In [ ]:
agg = df.groupby(['algorithm', 'pref_idx'])[
    ['util', 'access', 'stab', 'cog_balance', 'lifo']
].mean().reset_index()

obj_pairs = [
    ('util', 'access', 'Utilisation', 'Access Efficiency'),
    ('util', 'stab',   'Utilisation', 'Stability'),
    ('util', 'cog_balance', 'Utilisation', 'CoG Balance'),
    ('access', 'stab', 'Access Efficiency', 'Stability'),
    ('stab', 'cog_balance', 'Stability', 'CoG Balance'),
    ('util', 'lifo',   'Utilisation', 'LIFO Compliance'),
]
morl_pts = agg[agg.algorithm == 'morl_pct']
h_pts = agg[agg.algorithm != 'morl_pct']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (xa, ya, xl, yl) in zip(axes.flat, obj_pairs):
    ax.scatter(morl_pts[xa], morl_pts[ya], s=42, c='#2e7d32', alpha=0.75,
               label='MORL-PCT (preference sweep)', edgecolor='black', linewidth=0.4)
    colors = {'bl': '#1a4d7a', 'extreme_points': '#2a6aa6', 'baf': '#5a8bc9',
              'bssf': '#94b8d9', 'blsf': '#c5d8e6'}
    for hcode in HEURISTICS:
        sub = h_pts[h_pts.algorithm == hcode]
        ax.scatter(sub[xa], sub[ya], s=140, c=colors[hcode], marker='^',
                   label=hcode, edgecolor='black', linewidth=0.7, zorder=5)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc='best')
fig.suptitle('Pareto frontiers — MORL policy sweeps a surface; heuristics are single points',
             fontweight='bold')
plt.tight_layout(); plt.savefig(OUT/'pareto_2d.png', dpi=140, bbox_inches='tight'); plt.show()

## 6. 3D Pareto surface — util × access × stability

In [ ]:
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(morl_pts['util'], morl_pts['access'], morl_pts['stab'],
           s=42, c='#2e7d32', alpha=0.75, label='MORL-PCT (preference sweep)')
colors = {'bl':'#1a4d7a','extreme_points':'#2a6aa6','baf':'#5a8bc9','bssf':'#94b8d9','blsf':'#c5d8e6'}
for hcode in HEURISTICS:
    sub = h_pts[h_pts.algorithm == hcode]
    ax.scatter(sub['util'], sub['access'], sub['stab'], s=180, c=colors[hcode],
               marker='^', edgecolor='black', linewidth=0.8, label=hcode)
ax.set_xlabel('Utilisation'); ax.set_ylabel('Access Efficiency'); ax.set_zlabel('Stability')
ax.set_title('3D Pareto surface (3 of 5 objectives shown)')
ax.legend(fontsize=8, loc='upper left'); ax.view_init(elev=22, azim=42)
plt.tight_layout(); plt.savefig(OUT/'pareto_3d.png', dpi=140, bbox_inches='tight'); plt.show()

## 7. Where does MORL DOMINATE every heuristic?

In [ ]:
import numpy as np
obj_cols = ['util', 'access', 'stab', 'cog_balance', 'lifo']

def dominates(a, b):
    return all(a[c] >= b[c] for c in obj_cols) and any(a[c] > b[c] for c in obj_cols)

h_points = h_pts[obj_cols].values
h_codes = h_pts['algorithm'].values
morl_dominating = []
for _, m_row in morl_pts.iterrows():
    pi = int(m_row['pref_idx'])
    m_vals = {c: m_row[c] for c in obj_cols}
    beaten = []
    for h_vals, hc in zip(h_points, h_codes):
        h_dict = dict(zip(obj_cols, h_vals))
        if dominates(m_vals, h_dict):
            beaten.append(hc)
    if beaten:
        morl_dominating.append({'pref_idx': pi, 'preference': PREF_GRID[pi],
                                 **m_vals, 'dominates_heuristics': beaten})

if morl_dominating:
    print(f'MORL preference points that strictly Pareto-dominate at least one heuristic: {len(morl_dominating)}/{len(PREF_GRID)}')
    for d in morl_dominating[:10]:
        print(f"  pref {d['pref_idx']:>2}: util={d['util']:.3f} ae={d['access']:.3f} stab={d['stab']:.3f}  beats: {d['dominates_heuristics']}")
else:
    print('No strictly-dominating points (try more training / different preference grid)')

dom_df = pd.DataFrame(morl_dominating)
dom_df.to_csv(OUT/'morl_dominating_points.csv', index=False)

## 8. Per-objective leader table

In [ ]:
leaderboard = []
for obj in obj_cols:
    best_heur = h_pts.loc[h_pts[obj].idxmax()]
    best_morl = morl_pts.loc[morl_pts[obj].idxmax()]
    leaderboard.append({
        'objective': obj,
        'best_heuristic': best_heur['algorithm'],
        'heuristic_value': round(float(best_heur[obj]), 4),
        'best_morl_pref': PREF_GRID[int(best_morl['pref_idx'])],
        'morl_value': round(float(best_morl[obj]), 4),
        'morl_lead': round(float(best_morl[obj] - best_heur[obj]), 4),
    })
lb_df = pd.DataFrame(leaderboard)
lb_df.to_csv(OUT/'multi_metric_summary.csv', index=False)
print(lb_df.to_string(index=False))

## 9. Bundle everything

In [ ]:
import shutil
zip_path = '/kaggle/working/eval_morl_results.zip' if os.path.isdir('/kaggle/working') else 'eval_morl_results.zip'
shutil.make_archive(zip_path[:-4], 'zip', OUT)
print(f'bundled at {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:<32}  {f.stat().st_size/1024:>6.1f} KB')